# 03 — Baseline de popularité

**Objectif :** établir une référence reproductible et un split sans fuite par requête.

**Entrée :** `data/processed/train_clean.csv`.  
**Sorties :** split train/validation et métriques de popularité.  
**Dépendance :** notebook 02.  
**Temps estimé :** moins d'une minute.  
**Ressources :** CPU uniquement.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import yaml
from sklearn.model_selection import GroupShuffleSplit


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.metrics import evaluate_rankings

CONFIG = yaml.safe_load((ROOT / "configs" / "default.yaml").read_text(encoding="utf-8"))
SEED = int(CONFIG["project"]["seed"])
TOP_K = int(CONFIG["project"]["top_k"])
PROCESSED_DIR = ROOT / CONFIG["paths"]["processed_data"]
REPORTS_DIR = ROOT / CONFIG["paths"]["reports"]

train = pd.read_csv(PROCESSED_DIR / "train_clean.csv", dtype={"sku": str})
assert {"query_key", "sku"}.issubset(train.columns)
print(f"Observations={len(train):,}, requêtes={train['query_key'].nunique():,}")

## Split groupé par requête

Une requête normalisée appartient entièrement au train ou à la validation. Cela mesure la
capacité de généralisation vers des requêtes jamais vues et empêche la mémorisation de fuir.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, validation_idx = next(splitter.split(train, groups=train["query_key"]))
fit_frame = train.iloc[train_idx].copy()
validation = train.iloc[validation_idx].copy()

assert set(fit_frame["query_key"]).isdisjoint(validation["query_key"])
fit_frame.to_csv(PROCESSED_DIR / "fit_clicks.csv", index=False)
validation.to_csv(PROCESSED_DIR / "validation_clicks.csv", index=False)

print(f"Fit={len(fit_frame):,}; validation={len(validation):,}")

## Top 5 global

In [ ]:
global_top = fit_frame["sku"].value_counts().index.astype(str).tolist()
catalog_skus = train["sku"].drop_duplicates().astype(str).tolist()
global_top.extend(sku for sku in catalog_skus if sku not in global_top)
global_top = global_top[:TOP_K]
assert len(global_top) == TOP_K
assert len(global_top) == len(set(global_top))
print("Top global :", global_top)

actual_by_query = validation.groupby("query_key")["sku"].agg(lambda values: set(map(str, values)))
predictions = [global_top for _ in actual_by_query]
metrics = evaluate_rankings(actual_by_query.tolist(), predictions, TOP_K)
report = {
    "model": "global_popularity",
    "split": "group_shuffle_by_query_key",
    "fit_rows": int(len(fit_frame)),
    "validation_rows": int(len(validation)),
    "validation_queries": int(len(actual_by_query)),
    "top_skus": global_top,
    **metrics,
}

(REPORTS_DIR / "metrics_popularity.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Conclusion

Cette baseline garantit toujours cinq résultats et servira de fallback aux modèles suivants.